# Truncated Records in WARC Files (August 2019)

This metrics are based on WARC files from the August 2019 crawl (CC-MAIN-2019-35) after fixes and improvements in the WARC writer, cf. [cc/nutch#10](//github.com/commoncrawl/nutch/issues/10) and [metrics based on October 2018 WARC files](./cc-main-2018-43-single-warc-file.ipynb).

Truncated records are extracted using cc-pyspark, see [cc-pyspark#9](//github.com/commoncrawl/cc-pyspark/pull/9) from 100 randomly selected WARC files of the August 2019 crawl listed in [CC-MAIN-2019-35-warc.paths](./data/CC-MAIN-2019-35-warc.paths).

In [ ]:
import pandas

data = pandas.read_csv('data/truncated-records-CC-MAIN-2019-35-warc-100.csv')

Verify the solution of ([NUTCH-2729](https://issues.apache.org/jira/browse/NUTCH-2729)): all records having exactly the size of the content limit (1 MiB) are likely to be marked as truncated because the probability that a content payload is exactly 1 MiB is low.

There are no unmarked records in the analyzed set:

In [ ]:
data['truncated_reason'].value_counts(dropna=False)

Let's only look at pages/documents truncated because of the length limit

In [ ]:
tr = data[data['truncated_reason'] == 'length']

and check, first, how long the records are (all are exactly 1 MiB):

In [ ]:
tr['payload_length'].value_counts(dropna=False)

and, second, which content types are truncated:

In [ ]:
tr['identified_payload_type'].value_counts(dropna=False)

Now let's check records records truncated by another reason than the payload length:

In [ ]:
tr = data[data['truncated_reason'] != 'length']
tr['payload_length'].mean()/1024

The average size (after truncation) is significantly lower than 1 MiB, and the amount of HTML pages among the truncated document types is larger:

In [ ]:
tr['identified_payload_type'].value_counts(dropna=False)

Now let's look into WARC records sizes of all records truncated because of the length limit.

Because Common Crawl uses gzipped WARC files as primary archive format, it's actually more the size of the gzip-compressed WARC records which matters, not the size of the uncompressed content payload. Let's put the compressed and uncompressed lengths into relation for the various content types:

In [ ]:
data['count'] = 1
tr = data[data['truncated_reason'] == 'length']
tr = tr[['warc_record_length', 'warc_content_length', 'payload_length', 'identified_payload_type', 'count']]
r = tr.groupby(['identified_payload_type']).sum()
r = r[r['count'] > 3] # drop rare document types
r['% ratio'] = 100.0*r['warc_record_length']/r['warc_content_length']
for col in ['warc_record_length', 'warc_content_length', 'payload_length']:
    r[col] = r[col]/2**20 # show aggregated length in MiB
r

Note that the `warc_record_length` also includes the WARC headers while `warc_content_length` does not. That's why the compression ratio can exceed 100%.

Finally, a short look into super-large captures with an original `Content-Length` HTTP header exceeding 1 GiB:

In [ ]:
data[data['http_orig_content_length'] > 2**30].groupby(['identified_payload_type']).sum()